# Thành viên A – Bước 4 & 5: Huấn luyện U-Net++ 2D & Khôi phục Mask 3D (GPU L4)

**Khuyến nghị chọn Runtime trên Colab Pro:** `GPU L4` (24GB VRAM) + `RAM cao cấp`.

Quy trình thực hiện:
1. Mount Google Drive để lưu `best_model.pth`, log huấn luyện và 55 file mask 3D dự đoán.
2. Clone repo GitHub & cài thư viện.
3. Kéo dữ liệu sạch từ Kaggle Private Dataset `anhvunguyenquynh/brats2020-preprocessed-224-v11` về `/content/data`.
4. Huấn luyện U-Net++ 2D (`epochs=50`, `early_stopping=10`, `AMP=True`, chọn checkpoint theo Val 3D Dice).
5. Chạy dự đoán & khôi phục 55 ca test về lưới gốc (`<case_id>_pred_wt.nii.gz`) lưu thẳng vào Google Drive.

In [ ]:
# 1. Kiểm tra GPU & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo GitHub của nhóm & cài thư viện
REPO_URL = 'https://github.com/Q-Anh2610/brats-tumor-seg.git'  # <-- Sửa tên nick GitHub của bạn
!git clone {REPO_URL}
%cd brats-tumor-seg
!pip install -q -r requirements.txt

In [ ]:
# 3. Cấu hình Kaggle API & kéo dữ liệu đã tiền xử lý về /content/data
from google.colab import files
import os, glob

if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

DATA_DIR = '/content/data'
!mkdir -p {DATA_DIR}
!kaggle datasets download -d anhvunguyenquynh/brats2020-preprocessed-224-v11 -p {DATA_DIR} --unzip

# Tự động giải nén nếu Kaggle giữ file .tar bên trong
for tar_name in ['cases.tar', 'metadata.tar']:
    tar_path = os.path.join(DATA_DIR, tar_name)
    if os.path.exists(tar_path):
        !tar -xf {tar_path} -C {DATA_DIR}
        os.remove(tar_path)

print('Số file .npz:', len(glob.glob(f'{DATA_DIR}/cases/*.npz')))
print('Số file metadata .json:', len(glob.glob(f'{DATA_DIR}/metadata/*.json')))

In [ ]:
# 4. Huấn luyện U-Net++ 2D (lưu checkpoint & biểu đồ loss/dice thẳng vào Google Drive)
DRIVE_CKPT = '/content/drive/MyDrive/brats_midterm_unetpp'
!python -m src.train.train_unetpp \
    --config configs/train_config.yaml \
    --data-dir /content/data \
    --cache-dir /content/brats_npy_cache \
    --output-dir {DRIVE_CKPT}

In [ ]:
# Hiển thị đường cong huấn luyện (Loss & Val 3D Dice)
from IPython.display import Image, display
display(Image(filename=f'{DRIVE_CKPT}/training_curves.png'))

In [ ]:
# 5. Dự đoán và khôi phục mask 3D về lưới gốc cho 55 ca tập Test
DRIVE_PRED_TEST = '/content/drive/MyDrive/brats_midterm_unetpp/preds_test_3d'
!python -m src.train.predict_3d \
    --checkpoint {DRIVE_CKPT}/best_model.pth \
    --data-dir /content/data \
    --cache-dir /content/brats_npy_cache \
    --splits configs/splits.json \
    --split-name test \
    --out-dir {DRIVE_PRED_TEST}

In [ ]:
# Kiểm tra lại 55 file *_pred_wt.nii.gz vừa tạo và bảng quick check
import glob, pandas as pd
nii_files = sorted(glob.glob(f'{DRIVE_PRED_TEST}/*_pred_wt.nii.gz'))
print('Tổng số file 3D test đã xuất:', len(nii_files))
display(pd.read_csv(f'{DRIVE_PRED_TEST}/pred_summary_test.csv').head(10))